In [1]:
import sys
sys.path.append("../../")
import pandas as pd
import matplotlib.pyplot as plt

In [7]:
# data = pd.read_csv(r"../data/stroke_pre/stroke.csv")
data = pd.read_csv('data/diabetes_pre/diabetes.csv')
# data = pd.read_csv('../data/wids_pre/wids.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'data/diabetes_pre/diabetes.csv'

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1883 entries, 0 to 1882
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   gender             1883 non-null   int64  
 1   age                1883 non-null   int64  
 2   hypertension       1883 non-null   int64  
 3   heart_disease      1883 non-null   int64  
 4   ever_married       1883 non-null   int64  
 5   work_type          1883 non-null   int64  
 6   Residence_type     1883 non-null   int64  
 7   avg_glucose_level  1883 non-null   float64
 8   bmi                1883 non-null   float64
 9   smoking_status     1883 non-null   int64  
 10  stroke             1883 non-null   int64  
dtypes: float64(2), int64(9)
memory usage: 161.9 KB


In [ ]:
from callmefair.search.fair_search import BiasSearch

# dfbias = BiasSearch(data, 'DiagPeriodL90D', ['patient_age', 'race', 'payer', 'state_privileged', 'neighborhood_education'], n_threads=32) 
#dfbias = BiasSearch(data, 'stroke', ['age', 'gender', 'ever_married', 'Residence_type'], n_threads=10) 
dfbias = BiasSearch(data, 'readmitted', ['age', 'gender', 'race'], n_threads=16)

In [ ]:
# Evaluate all supported models programmatically
models = ['lr','mlp','xgb','cat','lgbm','tabtransformer']
results = {}
for m in models:
    table, printable = dfbias.evaluate_average(model_name=m)
    results[m] = table
    try:
        display(printable)
    except Exception:
        print(printable)
print(f'Completed models: {list(results.keys())}')

Training lr models (16 processes):   0%|          | 0/10 [00:39<?, ?it/s]


[Executor Error] Task failed in parallel execution
Backend: process | Workers: 16
Function: process_wrapper_training
Args (truncated repr): (      gender  age  hypertension  heart_disease  ever_married  work_type  \
0          1    0             0              0             1          0   
1          0    0             0              0             1          0   
2          1    0             1              0             1          0   
3          1    0             0              0             1          0   
4          1    1             0              1             1          0   
...      ...  ...           ...            ...   
Exception: KeyError: "['readmitted'] not in index"
Traceback:
concurrent.futures.process._RemoteTraceback: 
"""
Traceback (most recent call last):
  File "c:\Users\enemy\anaconda3\Lib\concurrent\futures\process.py", line 263, in _process_worker
    r = call_item.fn(*call_item.args, **call_item.kwargs)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

KeyError: "['readmitted'] not in index"

In [ ]:
table, printable = dfbias.evaluate_combination_average('age','race')
printable

In [ ]:
table, printable = dfbias.evaluate_combination_average('ever_married','gender')
printable

In [ ]:
table, printable = dfbias.evaluate_combinations()
printable

In [ ]:
tab1, tab2 = tab_lr, table[1:]
tab_final = [*tab1, *tab2]
print(tab_final)

In [ ]:
import plotly.graph_objects as go


def process_array(data, group_color):
    """Process 2D array into plot components with border coloring"""
    attributes = [d[0] for d in data[1:]]
    raw_scores = [d[1] for d in data[1:]]
    normalized = [d[2] for d in data[1:]]
    
    # Determine border colors based on normalized scores
    border_colors = ['red' if n == 1 else 'green' for n in normalized]
    
    return {
        'attributes': attributes,
        'raw': raw_scores,
        'border_colors': border_colors,
        'color': group_color
    }

# Process both datasets
group1 = process_array(tab_lr, '#1f77b4')  # Blue
group2 = process_array(table, '#ff7f0e')  # Orange

# Calculate x positions with gap between groups
x1 = list(range(len(group1['attributes'])))
x2 = [len(group1['attributes']) + 1 + i for i in range(len(group2['attributes']))]

# Create bar traces with conditional borders
trace1 = go.Bar(
    x=x1,
    y=group1['raw'],
    name='Group 1',
    marker=dict(
        color=group1['color'],
        line=dict(
            color=group1['border_colors'],
            width=3  # Thicker border for visibility
        )
    )
)

trace2 = go.Bar(
    x=x2,
    y=group2['raw'],
    name='Group 2',
    marker=dict(
        color=group2['color'],
        line=dict(
            color=group2['border_colors'],
            width=3
        )
    )
)

# Create figure
fig = go.Figure([trace1, trace2])

# Calculate axis settings
tickvals = x1 + x2
ticktext = group1['attributes'] + group2['attributes']
separator_pos = len(group1['attributes'])  # Position between groups

# Update layout with visual separation
fig.update_layout(
    xaxis=dict(
        tickvals=tickvals,
        ticktext=ticktext,
        title='Attributes',
        showgrid=False
    ),
    yaxis=dict(title='Raw Fairness Score'),
    title='Fairness Analysis with Normalization Indicators',
    bargap=0.25,
    shapes=[dict(
        type='line',
        xref='x',
        yref='paper',
        x0=separator_pos,
        y0=0,
        x1=separator_pos,
        y1=1,
        line=dict(color='gray', width=2, dash='dot')
    )]
)

fig.show()

In [ ]:
import plotly.graph_objects as go
import numpy as np

def create_fairness_visualization(attributes, raw_scores, normalized_scores, model_names, title="Attribute Fairness Impact"):
    """Create a visualization of fairness scores with density and depth cues."""
    fig = go.Figure()
    
    # Calculate average raw scores for sorting
    avg_raw_scores = [sum(scores) / len(scores) for scores in raw_scores]
    sort_indices = np.argsort(avg_raw_scores)
    
    # Sort attributes and scores
    sorted_attributes = [attributes[i] for i in sort_indices]
    sorted_raw_scores = [raw_scores[i] for i in sort_indices]
    sorted_norm_scores = [normalized_scores[i] for i in sort_indices]
    y_values = sorted_attributes
    n_attr = len(sorted_attributes)
    
    # Global min/max for x-axis and background shapes
    global_min = min(min(scores) for scores in sorted_raw_scores) - 0.2
    global_max = max(max(scores) for scores in sorted_raw_scores) + 0.2
    
    # Add subtle background bands per attribute (improves readability)
    for i in range(n_attr):
        fig.add_shape(
            type='rect',
            xref='x', yref='y',
            x0=global_min, x1=global_max,
            y0=i - 0.45, y1=i + 0.45,
            fillcolor='rgba(245, 245, 250, 0.8)',
            line=dict(width=0),
            layer='below'
        )
    
    # Add a soft diagonal "shadow" polygon behind everything to give a light 3D feel
    fig.add_shape(
        type='path',
        xref='x', yref='y',
        path=(
            f"M {global_min} {-0.5} "
            f"L {global_max + 0.15 * (global_max - global_min)} {-0.3} "
            f"L {global_max + 0.15 * (global_max - global_min)} {n_attr - 0.5} "
            f"L {global_min} {n_attr - 0.7} Z"
        ),
        fillcolor='rgba(210, 210, 230, 0.25)',
        line=dict(width=0),
        layer='below'
    )
    
    # Add per-model scatter points and violins
    for i, (attr, raw_list, norm_list) in enumerate(zip(sorted_attributes, sorted_raw_scores, sorted_norm_scores)):
        # Per-model points
        for j, (raw_val, norm_val) in enumerate(zip(raw_list, norm_list)):
            fig.add_trace(go.Scatter(
                x=[raw_val],
                y=[i],
                mode='markers',
                marker=dict(
                    size=10,
                    color=norm_val,
                    colorscale='RdYlGn_r',
                    line=dict(width=0),
                    opacity=0.9
                ),
                name=model_names[j],
                legendgroup=model_names[j],
                showlegend=(i == 0),
                hoverinfo='text',
                text=f"{attr} ({model_names[j]}): {raw_val:.3f}"
            ))
        
        # Violin based directly on the raw fairness scores for this attribute
        fig.add_trace(go.Violin(
            x=raw_list,
            y=[i] * len(raw_list),
            box_visible=False,
            points=False,
            line_color='rgba(120, 120, 120, 0.3)',
            fillcolor='rgba(180, 180, 180, 0.25)',
            width=0.6,
            side='both',
            orientation='h',
            showlegend=False,
            hoverinfo='none'
        ))
        
        # Highlight mean raw score
        mean_raw = sum(raw_list) / len(raw_list)
        mean_norm = sum(norm_list) / len(norm_list)
        fig.add_trace(go.Scatter(
            x=[mean_raw],
            y=[i],
            mode='markers',
            marker=dict(
                size=15,
                color=mean_norm,
                colorscale='RdBu',
                line=dict(width=1.2, color='black'),
                opacity=0.5
            ),
            showlegend=False,
            hoverinfo='text',
            text=f"{attr} (mean): {mean_raw:.3f}"
        ))
    
    # Configure layout with clearer background grid
    fig.update_layout(
        title=title,
        xaxis=dict(
            title='Fairness Score (impact on model output)',
            zeroline=False,
            showgrid=True,
            gridcolor='rgba(200, 200, 210, 0.6)',
            gridwidth=0.7,
            range=[global_min, global_max]
        ),
        yaxis=dict(
            title='',
            tickvals=list(range(n_attr)),
            ticktext=y_values,
            showgrid=False
        ),
        height=max(450, n_attr * 45),
        width=950,
        plot_bgcolor='rgba(255, 255, 255, 1)',
        margin=dict(l=170),
        legend=dict(title='Models'),
    )
    
    return fig

# Build visualization inputs programmatically from 'results'
model_names = models
base_table = results[model_names[0]]
attributes = [row[0] for row in base_table[1:]]
raw_scores = [[results[m][i+1][1] for m in model_names] for i in range(len(attributes))]
normalized_scores = [[results[m][i+1][2] for m in model_names] for i in range(len(attributes))]

fig = create_fairness_visualization(attributes, raw_scores, normalized_scores, model_names,
                                    title="Classifier Fairness Impact by Attribute")
fig.show()